In [8]:
from flask import Flask, jsonify, request
from flask_cors import CORS
import pymysql  # or import mysql.connector
import joblib
import pandas as pd
# calories, protien, sugar, fat, fiber, carbohydrates

In [ ]:
model = joblib.load("E:/College/MiniProject/Mini-Project-group-8/Program/SleepAnalysis.pkl")
# /home/kali/College/Mini/Job/SleepAnalysis.pkl
# E:/College/MiniProject/Mini-Project-group-8/Job/SleepAnalysis.pkl
db = pymysql.connect(
host = "localhost",
user = "root", #root
password = "root",
database = "mini"
)
cursor = db.cursor()

In [10]:
app = Flask(__name__)
cors = CORS(app, origins = '*')
@app.route("/submit", methods = ['GET', 'POST'] )
def submit():
    data = request.get_json()
    age = int(data['age'])
    bed_time = data['bedTime']
    wake_time = data['wakeTime']
    awakenings = float(data['awakenings'])
    caffeine = float(data['caffeine'])
    alcohol = float(data['alcohol'])
    smoking = "Yes" if data['smoking'].lower() == "yes" else "No"  # Store as Yes/No
    exercise = float(data['exercise'])
    REM = int(data['REM'])
    deep_sleep = int(data['deep_sleep'])


    smoking_numeric = 1 if smoking == "Yes" else 0
    sleep_duration = (float(wake_time.split(":")[0]) - float(bed_time.split(":")[0]) + 24) % 24

    userDataDF = pd.DataFrame({
        'Age': [age],
        'Sleep_duration': [sleep_duration],
        'REM_sleep_percentage': [REM], 
        'Deep_sleep_percentage': [deep_sleep], 
        'Awakenings': [awakenings],
        'Caffeine_consumption': [caffeine],
        'Alcohol_consumption': [alcohol],
        'Smoking_status': [smoking_numeric],  
        'Exercise_frequency': [exercise]
    })

    prediction = model.predict(userDataDF.to_numpy())
    sleep_efficiency = prediction[0]
    insert_query = """
        INSERT INTO sleep_data (age, bed_time, wake_time, awakenings, caffeine, alcohol, smoking, exercise, sleep_efficiency, REM_percentage, deep_sleep_percentage) 
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """
    values = (age, bed_time, wake_time, awakenings, caffeine, alcohol, smoking, exercise, sleep_efficiency, REM, deep_sleep)
    cursor.execute(insert_query, values)
    db.commit()
    return jsonify({'sleep_efficiency': prediction[0], "duration": sleep_duration })

@app.route("/diet", methods = ['GET', 'POST'])
def diet():
    pass

if __name__ == "__main__":
    app.run()

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [23/Mar/2025 18:18:52] "OPTIONS /submit HTTP/1.1" 200 -
[2025-03-23 18:18:52,305] ERROR in app: Exception on /submit [POST]
Traceback (most recent call last):
  File "c:\Users\HP\AppData\Local\anaconda3\Lib\site-packages\flask\app.py", line 1473, in wsgi_app
    response = self.full_dispatch_request()
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\HP\AppData\Local\anaconda3\Lib\site-packages\flask\app.py", line 882, in full_dispatch_request
    rv = self.handle_user_exception(e)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\HP\AppData\Local\anaconda3\Lib\site-packages\flask_cors\extension.py", line 176, in wrapped_function
    return cors_after_request(app.make_response(f(*args, **kwargs)))
                                                ^^^^^^^^^^^^^^^^^^
  File "c:\Users\HP\AppData\Local\anaconda3\Lib\site-packages\flask\app.py", line 880, in full_dispatch_request
    rv = self.dispatch_req